In [140]:
import requests
import time

In [160]:
token = "vk1.a.cKldW77LTQHbHCvfVvcMHUIlW6PHmz7OwNq09ba4LDZ9yGJtCCvQgQwUePB26831Qfea_lHG1Cz0ivyFsRZ5GnWGQO4Jql0SHN-wLFyqc8_r0G9oo54UIjWv7a8g5WDTnqHugFgG8F4U9WaYSn2oL46nfSiLTvtOnhwLZyQMcOfQgxa3L2Rd4NrPgrbINjiC"
user_id = 447450406

In [215]:
class VK:
    def __init__(self, token):
        self.token = token
        
        
    def get_user_info(self, user):
        param = {
            'access_token': token, 
             'user_ids': user, 
             # 'fields': 'about,books,bdate,career,common_count,connections,contacts,city,country,sex,schools,screen_name,interests,nickname,photo_100', 
             'fields': "nickname",
             'v': 5.131,
        }
        
        method = "users.get"

        rec = requests.get(url=f'https://api.vk.com/method/{method}', params=param)
        a = rec.json()
        res = ""
        try:
            res = a["response"]
        except KeyError as err:
            print(err)
            print(a)

        return res
    
    
    def get_followers(self, user_id):
        param = {'access_token': self.token, 
         'user_ids': user_id, 
         # 'fields': 'nickname,photo_100', 
         'v': 5.131}

        method = "users.getFollowers"

        rec = requests.get(url=f'https://api.vk.com/method/{method}', params=param)

        a = rec.json()

        ds = []
        try:
            for item in a["response"]["items"]:
    #             d = {}
    #             d["id"] = item["id"]
    #             # d["photo_100"] = item["photo_100"]
    #             ds.append(d)
                ds.append((item, user_id))
        except KeyError as err:
            print(err)
            print(a)
        return ds
    
    
    def get_subscription(self, user_id):
        param = {'access_token': self.token, 
                 'user_ids': user_id, 
                 # 'fields': 'nickname,photo_100', 
                 'v': 5.131}

        method = "users.getSubscriptions"

        rec = requests.get(url=f'https://api.vk.com/method/{method}', params=param)

        a = rec.json()

        d = []
        try:
            for item in a["response"]["users"]["items"]:
                d.append((user_id, item))
        except KeyError as err:
            print(err)
            print(a)
        return d
    
    
    def get_friends(self, user_id):
        param = {'access_token': self.token, 
                 'user_id': user_id, 
                 # 'fields': 'nickname,photo_100', 
                 
                 'v': 5.131}

        method = "friends.get"

        rec = requests.get(url=f'https://api.vk.com/method/{method}', params=param)

        a = rec.json()
        # print(a)
        b = []
        try:
            for item in a["response"]["items"]:
    #             d = {"id": item, 
    # #                  "first_name": item["first_name"],
    # #                  "last_name": item["last_name"], 
    # #                  "photo_100": item["photo_100"]
    #                 }
                b.append((user_id, item))
                b.append((item, user_id))
        except KeyError as err:
            print(err)
            print(a)
        return b
    
    def get_edges(self, user_id):
        return self.get_followers(user_id) + self.get_subscription(user_id) + self.get_friends(user_id)

In [234]:
class DFS:
    def __init__(self, vk):
        self.vk = vk
        
    def run(self, user_id, depth=1):
        a, b = self.reqursion(user_id, 1, depth, [user_id], [])
        return a, b
    
    def reqursion(self, user_id, level, final_level, already, ribbles):
        a = self.vk.get_edges(user_id)
        for i in a:
            if i not in ribbles and reversed(a) not in ribbles:
                ribbles.append(i)
        for i in a:
            if i[0] not in already:
                already.append(i[0])
                if level + 1 <= final_level:
                    b, c = self.reqursion(i[0], level + 1, final_level, already, ribbles)
                    already.extend(b)
                    for j in c:
                        if i not in ribbles and reversed(c) not in ribbles:
                            ribbles.append(i)
                  
            if i[1] not in already:
                already.append(i[1])
                if level + 1 <= final_level:
                    b, c = self.reqursion(i[1], level + 1, final_level, already, ribbles)
                    already.extend(b)
                    for j in c:
                        if i not in ribbles and reversed(c) not in ribbles:
                            ribbles.append(i)
            if len(already) > 1000000:
                break
        time.sleep(0.3)
        return already, ribbles
    
    def find_leaders(self, ribbles, lenght = 10):
        leaders = {}
        for i in ribbles:
            if i[1] not in leaders.keys():
                leaders.update({i[1]: [i[0]]})
            else:
                leaders.get(i[1]).append(i[0])

        leaders = sorted(leaders.items(), key=lambda x:len(x[1]))
        return leaders [-lenght:]
    
    def find_watchers(self, ribbles, lenght = 10):
        watchers = {}
        for i in ribbles:
            if i[0] not in watchers.keys():
                watchers.update({i[0]: [i[1]]})
            else:
                watchers.get(i[0]).append(i[1])

        watchers = sorted(watchers.items(), key=lambda x:len(x[1]))
        return leaders [-lenght:]
    
        
class BFS:
    def __init__(self, vk):
        self.vk = vk
        
    def run(self, user_id, depth=1):
        new = [user_id]
        old = []
        ribbles = []
        for _ in range(depth - 1):
            for i in new:
                a = self.vk.get_edges(i)
                newnew = []
                for j in a:
                    if j not in ribbles and reversed(j) not in ribbles:
                        ribbles.append(j)
                        
                    if j[0] not in old and j[0] not in new:
                        newnew.append(j[0])
                    if j[1] not in old and j[1] not in new:
                        newnew.append(j[1])
                for j in new:
                    if j not in old:
                        old.append(j)
                new = newnew  
                time.sleep(0.3)
        # int(ribbles)
        return old, ribbles
    
    
    def find_leaders(self, ribbles, lenght = 10):
        leaders = {}
        for i in ribbles:
            if i[1] not in leaders.keys():
                leaders.update({i[1]: [i[0]]})
            else:
                leaders.get(i[1]).append(i[0])

        leaders = sorted(leaders.items(), key=lambda x:len(x[1]))
        return leaders [-lenght:]
    
    def find_watchers(self, ribbles, lenght = 10):
        watchers = {}
        for i in ribbles:
            if i[0] not in watchers.keys():
                watchers.update({i[0]: [i[1]]})
            else:
                watchers.get(i[0]).append(i[1])

        watchers = sorted(watchers.items(), key=lambda x:len(x[1]))
        return watchers [-lenght:]
        

In [ ]:
vk = VK(token)
bfg = BFS(vk)
users, ribbles = bfg.run(447450406, 2)


with open("ribles.txt", "w")as f:
    f.write(users, ribbles)
        
print(bfg.find_leaders(ribbles, 10))

print(bfg.find_watchers(ribbles, 10))

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt

class GraphVisualization:
   
    def __init__(self):
        self.visual = []
        self.G = nx.DiGraph()
          
    def addEdge(self, a, b):
        temp = (a,b)
        self.visual.append(temp)
    
    def addEdges(self, a):
        for i in a:
            self.addEdge( i[0], i[1])

    def visualize(self):
        plt.figure(figsize=(20, 20), dpi=80)
        self.G.add_edges_from(self.visual)
        nx.draw_networkx(self.G)
        plt.show()

G = GraphVisualization()
G.addEdge(0, 1)
G.addEdge(0, 2)
G.addEdge(0, 3)
G.addEdge(1, 0)
G.addEdge(2, 0)
G.addEdge(3, 0)

G.addEdge(1, 4)
G.addEdge(1, 5)
G.addEdge(1, 6)
G.addEdge(2, 7)
G.addEdge(2, 8)
G.addEdge(2, 9)
G.addEdge(3, 10)
G.addEdge(3, 11)
G.addEdge(3, 12)

G.addEdge(4, 1)
G.addEdge(5, 1)
G.addEdge(6, 1)
G.addEdge(7, 2)
G.addEdge(8, 2)
G.addEdge(9, 2)
G.addEdge(10, 3)
G.addEdge(11,  3)
G.addEdge(12, 3)

k = 13
for j in range(4, 13):
    for i in range(3):
        G.addEdge(j, k + i + (j-4)*3)
        G.addEdge( k + i + (j-4)*3, j)
        

    
G.addEdge(10,6)
G.addEdge(6,10)
G.addEdge(35,4)
G.addEdge(4,35)
G.addEdge(9,10)

G.addEdge(39,40)
G.addEdge(40,39)

G.addEdge(29,1)
G.addEdge(1,8)
G.addEdge(22, 18)
G.addEdge(32, 2)

# G.visualize()
class VK2:
    def __init__(self, graph: GraphVisualization):
        self.graph = graph
    
    def get_edges(self, user_id):
        res = []
        for el in self.graph.visual:
            if user_id in el:
                res.append(el)
        return res
    
vk = VK2(G)
bfg = BFS(vk)
users, r = bfg.run(2, 3)



print(bfg.find_leaders(r, 10))

print(bfg.find_watchers(r, 10))

f = GraphVisualization()
f.addEdges(r)
f.visualize()
